In [0]:
# ==========================================================
# TSI - FASE 7: MLOPS
# CARGA DEL MODELO RANDOM FOREST V2
# ==========================================================

import mlflow
import mlflow.sklearn

RUN_ID = "3c49d369bc114c10812d0cafcdbc79da"

print("=== CARGA DEL MODELO DESDE MLFLOW ===")

# 1. Recuperar información del experimento
run = mlflow.get_run(RUN_ID)

assert run.info.status == "FINISHED"
assert run.data.params["version_tsi"] == "V2"

# 2. Construir URI del modelo registrado
modelo_uri = f"runs:/{RUN_ID}/modelo"

# 3. Cargar modelo entrenado
modelo_tsi = mlflow.sklearn.load_model(modelo_uri)

print("\n=== MODELO RECUPERADO ===")
print(f"Run ID: {RUN_ID}")
print(f"Tipo: {type(modelo_tsi).__name__}")
print(f"URI: {modelo_uri}")

# 4. Mostrar configuración
print("\n=== CONFIGURACIÓN ===")

print(f"Árboles: {modelo_tsi.n_estimators}")
print(f"Profundidad máxima: {modelo_tsi.max_depth}")
print(f"Muestras mínimas por hoja: {modelo_tsi.min_samples_leaf}")

print("\nMODELO CARGADO CORRECTAMENTE")
print("No se entrenaron modelos.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - MLOPS
# VERIFICAR TABLAS DISPONIBLES EN UNITY CATALOG
# ==========================================================

print("=== TABLAS DISPONIBLES EN WORKSPACE.TSI ===")

tablas = spark.sql("SHOW TABLES IN workspace.tsi")

display(
    tablas.filter(
        "lower(tableName) LIKE '%pems%' OR "
        "lower(tableName) LIKE '%tsi%' OR "
        "lower(tableName) LIKE '%international%'"
    )
)

print("VERIFICACIÓN COMPLETADA")

In [0]:
# ==========================================================
# TSI - MLOPS
# RECUPERACIÓN DEL CONJUNTO DE PRUEBA V2
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd

print("=== RECONSTRUCCIÓN TEST V2 ===")

# 1. Recuperar horarios del test original
test_v1 = spark.table("workspace.tsi.pems_bay_test_v1")

horas_test = test_v1.select("hora").distinct()

# 2. Recuperar observaciones PEMS-BAY
datos = spark.table("workspace.tsi.international_modeling")

datos = datos.filter(
    (F.col("fuente") == "pems_bay") &
    (F.col("valor_principal") > 0)
)

# 3. Limitar lectura al periodo necesario
inicio_test = test_v1.agg(F.min("hora")).first()[0]

datos = datos.filter(
    F.col("timestamp") >= F.lit(inicio_test)
)

# 4. Aplicar normalización V2
datos = datos.withColumn(
    "tsi_v2",
    F.greatest(
        F.lit(0.0),
        F.least(
            F.lit(100.0),
            100 * (
                (F.col("valor_principal") - 41.5)
                / (69.8 - 41.5)
            )
        )
    )
)

# 5. Agregación horaria
datos = datos.withColumn(
    "hora",
    F.date_trunc("hour", F.col("timestamp"))
)

horario = datos.groupBy("hora").agg(
    F.avg("tsi_v2").alias("tsi_promedio"),
    F.stddev("tsi_v2").alias("tsi_desviacion"),
    F.count("*").alias("observaciones"),
    F.countDistinct("segmento_id").alias("segmentos")
)

# 6. Construir pares de horas consecutivas
ventana = Window.orderBy("hora")

horario = (
    horario
    .withColumn(
        "hora_siguiente",
        F.lead("hora").over(ventana)
    )
    .withColumn(
        "tsi_siguiente",
        F.lead("tsi_promedio").over(ventana)
    )
)

horario = horario.filter(
    F.unix_timestamp("hora_siguiente")
    - F.unix_timestamp("hora") == 3600
)

# 7. Conservar únicamente horas del test original
df_test_v2 = horario.join(
    horas_test,
    on="hora",
    how="inner"
)

# 8. Convertir a pandas
df_test_v2_pd = df_test_v2.orderBy("hora").toPandas()

df_test_v2_pd["hora"] = pd.to_datetime(
    df_test_v2_pd["hora"]
)

df_test_v2_pd["hora_siguiente"] = pd.to_datetime(
    df_test_v2_pd["hora_siguiente"]
)

df_test_v2_pd["hora_dia"] = (
    df_test_v2_pd["hora"].dt.hour
)

df_test_v2_pd["dia_semana"] = (
    df_test_v2_pd["hora"].dt.dayofweek
)

# 9. Validar
total = len(df_test_v2_pd)

print(f"\nRegistros recuperados: {total:,}")
print(f"Primera hora: {df_test_v2_pd['hora'].min()}")
print(f"Última hora: {df_test_v2_pd['hora'].max()}")

assert total == 869, (
    f"Se esperaban 869 registros, pero se encontraron {total}"
)

print("\nTEST V2 RECUPERADO CORRECTAMENTE")
print("Sin reentrenamiento.")
print("Sin modificaciones a Delta Lake.")

In [0]:
# ==========================================================
# TSI - MLOPS
# INFERENCIA V2 + ALMACENAMIENTO EN DELTA LAKE
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pyspark.sql import functions as F

TABLA = "workspace.tsi.pems_bay_predicciones_v2"

FEATURES = [
    "tsi_promedio",
    "tsi_desviacion",
    "observaciones",
    "segmentos",
    "hora_dia",
    "dia_semana"
]

print("=== INFERENCIA RANDOM FOREST V2 ===")

# 1. Preparar datos
datos = df_test_v2_pd.copy()

X = datos[FEATURES].astype("float64")

assert len(datos) == 869
assert not X.isna().any().any()
assert np.isfinite(X.to_numpy()).all()

# 2. Generar predicciones
datos["tsi_predicho"] = modelo_tsi.predict(X)

# 3. Calcular errores
datos["error_absoluto"] = (
    datos["tsi_siguiente"] - datos["tsi_predicho"]
).abs()

datos["error_cuadrado"] = (
    datos["tsi_siguiente"] - datos["tsi_predicho"]
) ** 2

mae = mean_absolute_error(
    datos["tsi_siguiente"],
    datos["tsi_predicho"]
)

rmse = np.sqrt(mean_squared_error(
    datos["tsi_siguiente"],
    datos["tsi_predicho"]
))

print("\n=== MÉTRICAS DE INFERENCIA ===")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

# 4. Verificar reproducción de resultados MLflow
run = mlflow.get_run(RUN_ID)

mae_mlflow = run.data.metrics["mae_test"]
rmse_mlflow = run.data.metrics["rmse_test"]

assert abs(mae - mae_mlflow) < 0.01, (
    "El MAE no coincide con MLflow"
)

assert abs(rmse - rmse_mlflow) < 0.01, (
    "El RMSE no coincide con MLflow"
)

print("\nMétricas verificadas contra MLflow.")

# 5. Preparar tabla de predicciones
datos["modelo"] = "Random Forest V2"
datos["run_id"] = RUN_ID

columnas = [
    "hora",
    "hora_siguiente",
    "tsi_promedio",
    "tsi_siguiente",
    "tsi_predicho",
    "error_absoluto",
    "error_cuadrado",
    "modelo",
    "run_id"
]

df_guardar = spark.createDataFrame(
    datos[columnas].copy()
)

# 6. Guardar en Delta Lake
(
    df_guardar.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLA)
)

# 7. Confirmar almacenamiento
total = spark.table(TABLA).count()

assert total == 869

print("\n=== ALMACENAMIENTO COMPLETADO ===")
print(f"Tabla: {TABLA}")
print(f"Registros: {total:,}")
print("Modelo: Random Forest V2")
print("Predicciones verificadas y almacenadas.")

In [0]:
# ==========================================================
# TSI - FASE 7: MLOPS
# MONITOREO DE PREDICCIONES RANDOM FOREST V2
# ==========================================================

from pyspark.sql import functions as F

TABLA_PRED = "workspace.tsi.pems_bay_predicciones_v2"
TABLA_MON = "workspace.tsi.pems_bay_monitoreo_v2"

print("=== MONITOREO TSI V2 ===")

df = spark.table(TABLA_PRED)

# 1. Verificar integridad
total = df.count()

nulos = df.filter(
    F.col("tsi_predicho").isNull() |
    F.col("tsi_siguiente").isNull()
).count()

duplicados = (
    df.groupBy("hora")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert total == 869
assert nulos == 0
assert duplicados == 0

# 2. Métricas globales
resumen = df.agg(
    F.count("*").alias("registros"),
    F.avg("error_absoluto").alias("mae"),
    F.sqrt(F.avg("error_cuadrado")).alias("rmse"),
    F.max("error_absoluto").alias("error_maximo"),
    F.avg(
        (F.col("error_absoluto") > 5).cast("double")
    ).alias("proporcion_error_mayor_5")
)

print("\n=== MÉTRICAS GLOBALES ===")
display(resumen)

# 3. Errores por hora del día
por_hora = (
    df.withColumn("hora_dia", F.hour("hora"))
    .groupBy("hora_dia")
    .agg(
        F.count("*").alias("registros"),
        F.avg("error_absoluto").alias("mae"),
        F.sqrt(F.avg("error_cuadrado")).alias("rmse")
    )
    .orderBy("hora_dia")
)

print("\n=== ERRORES POR HORA ===")
display(por_hora)

# 4. Guardar monitoreo horario
(
    por_hora.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLA_MON)
)

# 5. Identificar mayores errores
print("\n=== CINCO MAYORES ERRORES ===")

display(
    df.orderBy(F.desc("error_absoluto"))
    .select(
        "hora",
        "tsi_siguiente",
        "tsi_predicho",
        "error_absoluto"
    )
    .limit(5)
)

print("\n=== VERIFICACIÓN FINAL ===")
print(f"Predicciones: {total:,}")
print(f"Nulos: {nulos}")
print(f"Horas duplicadas: {duplicados}")
print(f"Tabla monitoreo: {TABLA_MON}")
print("\nMONITOREO HISTÓRICO COMPLETADO")

In [0]:
# ==========================================================
# TSI - MLOPS
# VERIFICACIÓN FINAL DEL MONITOREO HISTÓRICO
# ==========================================================

from pyspark.sql import functions as F

TABLA_MON = "workspace.tsi.pems_bay_monitoreo_v2"
TABLA_PRED = "workspace.tsi.pems_bay_predicciones_v2"

print("=== VERIFICACIÓN DE TABLAS ===")

assert spark.catalog.tableExists(TABLA_MON)
assert spark.catalog.tableExists(TABLA_PRED)

monitoreo = spark.table(TABLA_MON)
predicciones = spark.table(TABLA_PRED)

print(f"Registros de predicción: {predicciones.count():,}")
print(f"Horas monitoreadas: {monitoreo.count()}")

print("\n=== CINCO HORAS CON MAYOR MAE ===")

display(
    monitoreo.orderBy(F.desc("mae")).limit(5)
)

print("\n=== CINCO PREDICCIONES CON MAYOR ERROR ===")

display(
    predicciones.orderBy(F.desc("error_absoluto"))
    .select(
        "hora",
        "tsi_siguiente",
        "tsi_predicho",
        "error_absoluto"
    )
    .limit(5)
)

print("\nVERIFICACIÓN DE MLOPS COMPLETADA")